In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [2]:
# Cargar bases de datos
b1 = pd.read_csv('hey_clientes.csv')
b2 = pd.read_csv('hey_productos.csv')
b3 = pd.read_csv('hey_transacciones.csv')
#b4 = pd.read_csv('dataset_50k_anonymized.csv')  # input, output, user_id

In [3]:
# ── Agrega Base2 por usuario ──────────────────────────────────────────
# Pivoteamos los productos para tener una fila por usuario
b2_pivot = b2.pivot_table(
    index="user_id",
    columns="tipo_producto",
    values="saldo_actual",
    aggfunc="sum",
    fill_value=0
).reset_index()

b2_extra = b2.groupby("user_id").agg(
    num_productos=("producto_id", "count"),
    utilizacion_media=("utilizacion_pct", "mean"),
    saldo_total=("saldo_actual", "sum"),
).reset_index()

In [4]:
# ── Agrega Base3 por usuario ──────────────────────────────────────────
b3_agg = b3.groupby("user_id").agg(
    total_transacciones=("transaccion_id", "count"),
    monto_total=("monto", "sum"),
    cashback_total=("cashback_generado", "sum"),
    pct_internacional=("es_internacional", "mean"),
    pct_atipico=("patron_uso_atipico", "mean"),
).reset_index()

In [6]:
# ── Merge final ───────────────────────────────────────────────────────
df = (b1
      .merge(b2_pivot,  on="user_id", how="left")
      .merge(b2_extra,  on="user_id", how="left")
      .merge(b3_agg,    on="user_id", how="left"))

df.fillna(0)
print(df.shape)  # (n_usuarios, n_features)


(15025, 41)


In [7]:
print(f"Dataset final: {df.shape}")
print(f"Variables disponibles: {len(df.columns)}")

Dataset final: (15025, 41)
Variables disponibles: 41


# Cargar los clusters de conversación

In [8]:
clusters_input  = pd.read_parquet('dataset_clusters_in.parquet')  # user_id, cluster
clusters_output = pd.read_parquet('dataset_clusters_output.parquet') # user_id, cluster

clusters_input.rename(columns={"cluster": "cluster_input"}, inplace=True)
clusters_output.rename(columns={"cluster": "cluster_output"}, inplace=True)

df = (df
      .merge(clusters_input,  on="user_id", how="left")
      .merge(clusters_output, on="user_id", how="left"))

# Preparar features para el modelo de satisfacción

In [12]:
from sklearn.preprocessing import LabelEncoder

# Variables a usar como predictoras
FEATURES = [
    "edad", "ingreso_mensual_mxn", "antiguedad_dias", "score_buro",
    "dias_desde_ultimo_login", "num_productos_activos",
    "es_hey_pro", "nomina_domiciliada", "recibe_remesas", "usa_hey_shop",
    "tiene_seguro", "patron_uso_atipico",
    "num_productos", "utilizacion_media", "saldo_total",
    "total_transacciones", "monto_total", "cashback_total",
    "pct_internacional", "pct_atipico",
    "conv_cluster_x", "conv_cluster_y",
]

TARGET = "satisfaccion_1_10"

# Codificar booleanos
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

print(df.columns)

X = df[FEATURES].fillna(0)
y = df[TARGET]

Index(['user_id', 'edad', 'sexo', 'estado', 'ciudad', 'nivel_educativo',
       'ocupacion', 'ingreso_mensual_mxn', 'antiguedad_dias', 'es_hey_pro',
       'nomina_domiciliada', 'canal_apertura', 'score_buro',
       'dias_desde_ultimo_login', 'preferencia_canal', 'satisfaccion_1_10',
       'recibe_remesas', 'usa_hey_shop', 'idioma_preferido', 'tiene_seguro',
       'num_productos_activos', 'patron_uso_atipico', 'credito_auto',
       'credito_nomina', 'credito_personal', 'cuenta_debito',
       'cuenta_negocios', 'inversion_hey', 'seguro_compras', 'seguro_vida',
       'tarjeta_credito_garantizada', 'tarjeta_credito_hey',
       'tarjeta_credito_negocios', 'num_productos', 'utilizacion_media',
       'saldo_total', 'total_transacciones', 'monto_total', 'cashback_total',
       'pct_internacional', 'pct_atipico', 'conv_id_x', 'conv_cluster_x',
       'conv_id_y', 'conv_cluster_y'],
      dtype='str')


In [15]:
print(f"Nulos en satisfaccion_1_10: {y.isna().sum()}")
print(f"Infinitos: {np.isinf(y).sum()}")

mask = y.notna() & np.isfinite(y)
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Filas válidas para entrenar: {len(y)}")

Nulos en satisfaccion_1_10: 2804
Infinitos: 0
Filas válidas para entrenar: 52781


# Entrenar modelo de satisfacción con XGBoost

In [13]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 2.1/101.7 MB 17.7 MB/s eta 0:00:06
   -- ------------------------------------- 6.6/101.7 MB 20.1 MB/s eta 0:00:05
   ---- ----------------------------------- 11.5/101.7 MB 21.4 MB/s eta 0:00:05
   ------ --------------------------------- 16.8/101.7 MB 22.5 MB/s eta 0:00:04
   --------- ------------------------------ 23.1/101.7 MB 24.2 MB/s eta 0:00:04
   ----------- ---------------------------- 29.1/101.7 MB 25.3 MB/s eta 0:00:03
   ------------- -------------------------- 34.9/101.7 MB 25.3 MB/s eta 0:00:03
   --------------- ------------------------ 39.6/101.7 MB 25.2 MB/s eta 0:00:03
   ----------------- ---------------------- 45.6/101.7 MB 25.2 MB/s eta 0:00:03
   ------------------- -------------------- 50.6/101.7 MB 25.2 MB/s eta 0:00:03
   ---------------------- ----------------- 56.4/101.7 MB 25.2 MB/s eta 0:00:02
   ------------------------ --------------- 62.4/10


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = XGBRegressor(n_estimators=300, max_depth=5,
                      learning_rate=0.05, random_state=42)
modelo.fit(X_train, y_train)

mae = mean_absolute_error(y_test, modelo.predict(X_test))
print(f"MAE: {mae:.3f}")  # Error promedio

MAE: 0.807


# Identificar qué variables más afectan la satisfacción

In [18]:
importances = pd.Series(modelo.feature_importances_, index=FEATURES)
top_features = importances.sort_values(ascending=False).head(10)
print(top_features)

utilizacion_media          0.324913
dias_desde_ultimo_login    0.295354
num_productos              0.054288
score_buro                 0.044526
nomina_domiciliada         0.032065
cashback_total             0.029035
total_transacciones        0.027202
monto_total                0.019946
recibe_remesas             0.018049
saldo_total                0.017580
dtype: float32


# Definir las reglas de sugerencia por cluster

In [ ]:
# Descripción de cada cluster (la defines explorando los centroides del PCA)
CLUSTER_INPUT_DESC = {
    0: "usuario que pregunta sobre saldos y movimientos",
    1: "usuario que pregunta sobre productos y beneficios",
    2: "usuario que reporta problemas o quejas",
}

CLUSTER_OUTPUT_DESC = {
    0: "respuestas cortas y directas",
    1: "respuestas explicativas con contexto",
    2: "respuestas con opciones y alternativas",
}

# Reglas de sugerencia basadas en variables del modelo
def generar_sugerencia(row, top_features):
    sugerencias = []
    feat_principal = top_features.index[0]

    if feat_principal == "dias_desde_ultimo_login" and row["dias_desde_ultimo_login"] > 30:
        sugerencias.append("Invitar al usuario a explorar su app con recordatorio de funciones nuevas.")

    if feat_principal == "utilizacion_media" and row.get("utilizacion_media", 0) > 0.8:
        sugerencias.append("Ofrecer aumento de límite o productos de crédito complementarios.")

    if feat_principal == "cashback_total" and row.get("cashback_total", 0) == 0:
        sugerencias.append("Informar sobre el programa de cashback que aún no ha aprovechado.")

    if row.get("tiene_seguro", 0) == 0:
        sugerencias.append("Presentar beneficios del seguro de vida o compras.")

    if not sugerencias:
        sugerencias.append("Mantener la conversación con información sobre sus productos actuales.")

    return sugerencias

<!-- # 6.2 -->

# 6.2

In [71]:
# ── Umbrales de referencia (ajusta según tu negocio) ──────────────────
SCORE_BURO_BUENO     = 650
INGRESO_ALTO         = 25_000
UTILIZACION_ALTA     = 0.75
SALDO_INVERSION      = 5_000
ANTIGUEDAD_FIEL      = 365
SATISFACCION_BAJA    = 6.0

# ── Productos disponibles para recomendar ─────────────────────────────
PRODUCTOS_CREDITO = [
    "tarjeta_credito_hey",
    "tarjeta_credito_negocios",
    "tarjeta_credito_garantizada",
    "credito_personal",
    "credito_nomina",
    "credito_auto",
]

PRODUCTOS_AHORRO = [
    "inversion_hey",
    "cuenta_negocios",
]

PRODUCTOS_SEGURO = [
    "seguro_vida",
    "seguro_compras",
]

In [92]:
def obtener_productos_activos(user_id: str, base2: pd.DataFrame) -> list:
    """Devuelve la lista de tipo_producto que ya tiene el usuario."""
    return base2[base2["user_id"] == user_id]["tipo_producto"].unique().tolist()


def recomendar(user_id: str, df: pd.DataFrame, base2: pd.DataFrame,
               modelo, top_features: pd.Series) -> str:

    # ── 1. Obtener perfil ─────────────────────────────────────────────
    row = df[df["user_id"] == user_id]
    if row.empty:
        return f"No se encontró el usuario '{user_id}'."
    row = row.iloc[0]

    productos_actuales = obtener_productos_activos(user_id, base2)

    # ── 2. Predecir satisfacción ──────────────────────────────────────
    x_user = pd.DataFrame([row[FEATURES]])
    x_user = x_user.replace([np.inf, -np.inf], np.nan).fillna(0)
    sat_pred = modelo.predict(x_user)[0]
    sat_real = row["satisfaccion_1_10"]

    # ── 3. Datos del usuario ──────────────────────────────────────────
    ingreso        = row.get("ingreso_mensual_mxn", 0)
    score          = row.get("score_buro", 0)
    antiguedad     = row.get("antiguedad_dias", 0)
    utilizacion    = row.get("utilizacion_media", 0)
    saldo_total    = row.get("saldo_total", 0)
    cashback       = row.get("cashback_total", 0)
    tiene_seguro   = bool(row.get("tiene_seguro", 0))
    es_pro         = bool(row.get("es_hey_pro", 0))
    nomina         = bool(row.get("nomina_domiciliada", 0))
    dias_login     = row.get("dias_desde_ultimo_login", 0)
    cl_in          = int(row.get("conv_cluster_x", 0))
    cl_out         = int(row.get("conv_cluster_y", 0))

    recomendaciones = []
    alertas         = []

    # ── 4. Recomendaciones de crédito ─────────────────────────────────
    productos_credito_faltantes = [
        p for p in PRODUCTOS_CREDITO if p not in productos_actuales
    ]

    if score >= SCORE_BURO_BUENO and ingreso >= INGRESO_ALTO:
        if "tarjeta_credito_hey" not in productos_actuales:
            recomendaciones.append(
                "Tarjeta de crédito Hey — perfil crediticio sólido "
                f"(score {score}, ingreso ${ingreso:,.0f}/mes)."
            )
        if "tarjeta_credito_negocios" not in productos_actuales and ingreso >= 40_000:
            recomendaciones.append(
                "Tarjeta de crédito Negocios — ingresos altos y buen historial."
            )

    if score >= SCORE_BURO_BUENO and "credito_auto" not in productos_actuales:
        recomendaciones.append(
            f"Crédito Auto — score buro {score} califica para condiciones preferenciales."
        )

    if nomina and "credito_nomina" not in productos_actuales:
        recomendaciones.append(
            "Crédito Nómina — nómina domiciliada activa, tasa preferencial disponible."
        )

    if "tarjeta_credito_garantizada" not in productos_actuales and score < SCORE_BURO_BUENO:
        recomendaciones.append(
            "Tarjeta de crédito Garantizada — para construir historial crediticio "
            f"(score actual: {score})."
        )

    # ── 5. Aumento de límite de crédito ───────────────────────────────
    if utilizacion >= UTILIZACION_ALTA and score >= SCORE_BURO_BUENO:
        recomendaciones.append(
            f"Aumento de límite de crédito — utilización actual alta "
            f"({utilizacion*100:.0f}%) con buen score. Candidato a revisión."
        )

    # ── 6. Recomendaciones de ahorro e inversión ──────────────────────
    if "inversion_hey" not in productos_actuales and saldo_total >= SALDO_INVERSION:
        recomendaciones.append(
            f"Inversión Hey — saldo disponible ${saldo_total:,.0f}. "
            "Puede generar rendimientos desde hoy."
        )

    if "cuenta_negocios" not in productos_actuales and ingreso >= 30_000:
        recomendaciones.append(
            "Cuenta Negocios — perfil de ingresos compatible con actividad empresarial."
        )

    # ── 7. Seguros ────────────────────────────────────────────────────
    if not tiene_seguro:
        if "seguro_vida" not in productos_actuales:
            recomendaciones.append(
                "Seguro de Vida — no cuenta con ningún seguro activo. "
                "Cotización disponible según perfil."
            )
        if "seguro_compras" not in productos_actuales:
            recomendaciones.append(
                "Seguro de Compras — protección para sus transacciones frecuentes."
            )

    # ── 8. Alertas de comportamiento ──────────────────────────────────
    if dias_login > 30:
        alertas.append(
            f"Sin actividad hace {int(dias_login)} días — enviar recordatorio "
            "de funciones nuevas o beneficios no utilizados."
        )

    if cashback == 0 and "tarjeta_credito_hey" in productos_actuales:
        alertas.append(
            "No ha generado cashback — puede no estar usando su tarjeta Hey "
            "para compras cotidianas."
        )

    if sat_pred < SATISFACCION_BAJA:
        alertas.append(
            f"Satisfacción estimada baja ({sat_pred:.1f}/10) — "
            "priorizar atención proactiva."
        )

    # ── 9. Perfil de conversación (clusters) ──────────────────────────
    estilo_msg = {
        0: "Responder de forma directa y concisa.",
        1: "Incluir contexto y beneficios detallados.",
        2: "Ofrecer alternativas y resolver dudas primero.",
    }.get(cl_out, "Estilo estándar.")

    tipo_consulta = {
        0: "Consultas sobre saldos y movimientos",
        1: "Interés en productos y beneficios",
        2: "Reportes de problemas o quejas",
    }.get(cl_in, "Consulta general")

    # ── 10. Armar respuesta ───────────────────────────────────────────
    

    salida = f"\n  RECOMENDACIONES\n"
    if recomendaciones:
        for i, r in enumerate(recomendaciones, 1):
            salida += f"\n  {i}. {r}"
    else:
        salida += "\n  El usuario ya cuenta con los productos adecuados a su perfil."

    #salida += f"\n  ALERTAS INTERNAS\n"

    #salida += f"\n"
    return salida

In [94]:
print(recomendar("USR-00093", df, b2, modelo, top_features))


  RECOMENDACIONES

  1. Tarjeta de crédito Garantizada — para construir historial crediticio (score actual: 314).
  2. Inversión Hey — saldo disponible $147,787. Puede generar rendimientos desde hoy.
  3. Seguro de Vida — no cuenta con ningún seguro activo. Cotización disponible según perfil.
  4. Seguro de Compras — protección para sus transacciones frecuentes.


# Hasta aqui

# Función principal del bot

In [38]:
def bot_personalizado(user_id: str):
    # 1. Obtener perfil del usuario
    row = df[df["user_id"] == user_id]
    #print(row.columns)
    if row.empty:
        return "Usuario no encontrado."
    row = row.iloc[0]

    # 2. Predecir satisfacción actual
    x_user = pd.DataFrame([row[FEATURES]])
    sat_pred = modelo.predict(x_user)[0]

    # 3. Obtener clusters de conversación
    cl_in  = int(row["conv_cluster_x"])
    cl_out = int(row["conv_cluster_y"])

    # 4. Generar sugerencias
    sugerencias = generar_sugerencia(row, top_features)

    # 5. Construir respuesta personalizada
    perfil_conv  = CLUSTER_INPUT_DESC.get(cl_in, "perfil desconocido")
    estilo_resp  = CLUSTER_OUTPUT_DESC.get(cl_out, "estilo estándar")

    respuesta = (
        f"[Bot interno — usuario: {user_id}]\n"
        f"Satisfacción estimada: {sat_pred:.1f}/10\n"
        f"Perfil de conversación: {perfil_conv}\n"
        f"Estilo de respuesta sugerido: {estilo_resp}\n\n"
        f"Sugerencias para mejorar satisfacción:\n"
    )
    for i, s in enumerate(sugerencias, 1):
        respuesta += f"  {i}. {s}\n"

    return respuesta



In [39]:
# ── Ejemplo de uso ────────────────────────────────────────────────────
print(bot_personalizado("USR-00003"))

[Bot interno — usuario: USR-00003]
Satisfacción estimada: 8.4/10
Perfil de conversación: usuario que reporta problemas o quejas
Estilo de respuesta sugerido: respuestas cortas y directas

Sugerencias para mejorar satisfacción:
  1. Presentar beneficios del seguro de vida o compras.



# Extra